# Bell State with SpinQit's Native API

This notebook explains `scripts/native_bell_state.py` step by step.

We will prepare a **Bell state** on the **Triangulum II** quantum computer using SpinQit's native `Circuit` interface.

## What is a Bell state?

A Bell state is a maximally entangled state of two qubits. The circuit below prepares:

$$|\Phi^+\rangle = \frac{1}{\sqrt{2}}\left(|00\rangle + |11\rangle\right)$$

Steps:
1. Apply a **Hadamard** gate on qubit 0 → superposition $(|0\rangle + |1\rangle)/\sqrt{2}$
2. Apply a **CNOT** with control qubit 0 and target qubit 1 → entanglement

```
q0: ──H──●──
         │
q1: ─────X──
```

## Imports

SpinQit provides the native circuit builder, gates, compiler, and backends (simulator or NMR device).

In [1]:
from spinqit import (
    Circuit,
    H,
    CX,
    BasicSimulatorConfig,
    NMRConfig,
    get_basic_simulator,
    get_compiler,
    get_nmr,
)

## Build the circuit

In the native API:
- create a `Circuit`
- allocate qubits with `allocateQubits`
- append gates with the `<<` operator

In [2]:
circ = Circuit()
q = circ.allocateQubits(2)

circ << (H, q[0])
circ << (CX, (q[0], q[1]))

print("Circuit created with 2 qubits.")

Circuit created with 2 qubits.


## Choose backend: simulator or NMR

Set `USE_SIMULATOR = True` to run locally. Set it to `False` and fill in the NMR settings to run on the Triangulum II hardware.

In [3]:
USE_SIMULATOR = True

# NMR settings (only used when USE_SIMULATOR is False)
NMR_IP = "127.0.0.1"
NMR_PORT = 8989
NMR_USERNAME = "your_username"
NMR_PASSWORD = "your_password"
NMR_TASK = "bell_state_task"
NMR_TASK_TYPE = "GHZ"
SHOTS = 1024

In [4]:
compiler = get_compiler("native")

if USE_SIMULATOR:
    device = get_basic_simulator()
    config = BasicSimulatorConfig()
    config.configure_shots(SHOTS)
    print("Using basic simulator.")
else:
    device = get_nmr()
    config = NMRConfig()
    config.configure_shots(SHOTS)
    config.configure_ip(NMR_IP)
    config.configure_port(NMR_PORT)
    config.configure_account(NMR_USERNAME, NMR_PASSWORD)
    config.configure_task(NMR_TASK, NMR_TASK_TYPE)
    print("Using NMR device.")

Using basic simulator.


## Compile and execute

The compiler translates the native circuit into an executable program for the selected backend.

In [5]:
exe = compiler.compile(circ, 0)
result = device.execute(exe, config)

## Results

- **`states`**: full state vector (available on simulators when there is no measurement)
- **`counts`**: number of times each bitstring was observed over many shots

For a Bell state without measurement, you should see amplitudes only on $|00\rangle$ and $|11\rangle$.

SpinQit uses **little-endian** bit ordering: the first character in a result string corresponds to qubit 0.

In [6]:
print("States:")
print(result.states)

print("\nCounts:")
print(result.counts)

States:
[(0.7071067811865476+0j), 0j, 0j, (0.7071067811865476+0j)]

Counts:
{'00': 512, '11': 512}


## Summary

| Step | Native SpinQit API |
|------|--------------------|
| Create circuit | `Circuit()` |
| Allocate qubits | `circ.allocateQubits(2)` |
| Add gates | `circ << (H, q[0])`, `circ << (CX, (q[0], q[1]))` |
| Compiler | `get_compiler("native")` |
| Run | `device.execute(exe, config)` |

The equivalent terminal script is `scripts/native_bell_state.py`.